# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [39]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [40]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['AADKTZOQUB', 'BPIYXFTTTF'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 1,  1,  4, 11, 20, 26, 15, 17, 21,  2],
       [ 2, 16,  9, 25, 24,  6, 20, 20, 20,  6]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  2, 21, 17, 15, 26, 20, 11,  4,  1],
       [ 0,  6, 20, 20, 20,  6, 24, 25,  9, 16]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 2, 21, 17, 15, 26, 20, 11,  4,  1,  1],
       [ 6, 20, 20, 20,  6, 24, 25,  9, 16,  2]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [41]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz = 27
        self.hidden = 128
        self.embedding = tf.keras.layers.Embedding(self.v_sz, 64)
        
        # 编码器
        self.encoder = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(self.hidden, return_sequences=True, return_state=True)
        )
        # 为了简化，使用 LSTM 替代 RNN，因为 LSTM 有更好的梯度流
        
        # 解码器 LSTM
        self.decoder_lstm = tf.keras.layers.LSTM(self.hidden, return_sequences=True, return_state=True)
        
        # 注意力层
        self.attention = tf.keras.layers.Attention()
        
        # 输出层
        self.dense = tf.keras.layers.Dense(self.v_sz)
    
    def call(self, enc_ids, dec_ids):
        # 编码
        enc_emb = self.embedding(enc_ids)
        enc_out, forward_h, forward_c, backward_h, backward_c = self.encoder(enc_emb)
        # 合并双向状态作为解码器初始状态
        state_h = tf.concat([forward_h, backward_h], axis=-1)
        state_c = tf.concat([forward_c, backward_c], axis=-1)
        
        # 解码器输入
        dec_emb = self.embedding(dec_ids)
        # 解码器初始状态
        dec_out, state_h, state_c = self.decoder_lstm(dec_emb, initial_state=[state_h, state_c])
        
        # 注意力：将编码器输出作为 value，解码器输出作为 query
        context = self.attention([dec_out, enc_out])
        # 拼接上下文和解码器输出
        dec_combined = tf.concat([dec_out, context], axis=-1)
        logits = self.dense(dec_combined)
        return logits
    
    def encode(self, enc_ids):
        enc_emb = self.embedding(enc_ids)
        enc_out, forward_h, forward_c, backward_h, backward_c = self.encoder(enc_emb)
        state_h = tf.concat([forward_h, backward_h], axis=-1)
        state_c = tf.concat([forward_c, backward_c], axis=-1)
        return enc_out, [state_h, state_c]
    
    def get_next_token(self, x, state, enc_out):
        # x: (batch,)
        # state: [h, c]
        h, c = state
        x_emb = self.embedding(x)  # (b, emb)
        # 扩展维度以符合 LSTM 输入 (b, 1, emb)
        x_emb = tf.expand_dims(x_emb, axis=1)
        # 解码一步
        dec_out, h, c = self.decoder_lstm(x_emb, initial_state=[h, c])
        # 注意力
        context = self.attention([dec_out, enc_out])
        dec_combined = tf.concat([dec_out, context], axis=-1)
        logits = self.dense(dec_combined)  # (b, 1, v_sz)
        logits = tf.squeeze(logits, axis=1)  # (b, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, [h, c]

# Loss函数以及训练逻辑

In [42]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [43]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

ValueError: in user code:

    File "C:\Users\ck\AppData\Local\Temp\ipykernel_32364\2008361797.py", line 11, in train_one_step  *
        logits = model(enc_x, dec_x)
    File "e:\Anaconda3\Lib\site-packages\keras\src\utils\traceback_utils.py", line 122, in error_handler  **
        raise e.with_traceback(filtered_tb) from None
    File "C:\Users\ck\AppData\Local\Temp\ipykernel_32364\2476188605.py", line 34, in call
        dec_out, state_h, state_c = self.decoder_lstm(dec_emb, initial_state=[state_h, state_c])

    ValueError: Exception encountered when calling LSTMCell.call().
    
    [1mDimensions must be equal, but are 256 and 128 for '{{node my_seq2_seq_model_7_1/lstm_3_1/lstm_cell_1/MatMul_1}} = MatMul[T=DT_FLOAT, grad_a=false, grad_b=false, transpose_a=false, transpose_b=false](my_seq2_seq_model_7_1/concat, my_seq2_seq_model_7_1/lstm_3_1/lstm_cell_1/Cast_1/ReadVariableOp)' with input shapes: [32,256], [128,512].[0m
    
    Arguments received by LSTMCell.call():
      • input

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [ ]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[False, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, False, False, True, True, True, True, True, True, True]
[('LGNVPEUAIFNLDIRQPMEY', 'YEMPQRIDLNFIAUEPVNGL'), ('ZFKNSOQGHINBCDJMOQXO', 'OXQOMJDCBNIHGQOSNKFZ'), ('ZXGDWLBRJMDSNYXIEMKJ', 'JKMEIXYNSDMJRBLWDGXZ'), ('KSEOFXPRMGUBHSJWIGYC', 'CYGIWJSHBUGMRPXFOESK'), ('HHMACLLSVMESLETLYJLZ', 'ZLJYLTELSEMVSLLCAMHH'), ('PVIABJTRCEQZFSJZLKQP', 'PQKLZJSFZQECRTJBAIVP'), ('PTUYDMQCQKZWBNCJRZBG', 'GBZRJCNBWZKQCQMDYUTP'), ('ZFZEWFPTWQFFSNNQLOFW', 'WFOLQNNSFFQWTPFWEZFZ'), ('NAHVTGYNHHEXOKMOEDRN', 'NRDEOMKOXEHHNYGTVHAN'), ('CTDDCDXRMIQDKCHODXVU', 'UVXDOHCKDQIMRXDCDDTL'), ('QWMRORZQQQTSJEHNVSZN', 'NZSVNHEJSTQQQZRORMWQ'), ('HTLTKWDCPXYCOREQGOUA', 'AUOGQEROCYXPCDWKTLTH'), ('DIQEGEZTNCFGNHHCXEDE', 'EDEXCHHNGFCNTZEGEQIJ'), ('HLGMERZQBOYYDQWMKKVE', 'EVKKMWQDYYOBQZREMGLH'), ('MCUCLWJZZVBMJFTNMAMU', 'UMAMNTFJMBVZZJWLCUCY'), ('UJIKEJYXYBBUSNIDOMBL', 'LBMODINSUBBYXYJEKIJU'), ('